In [ ]:
%pip install -r requirements.txt

In [ ]:
# 設定參數與專案資料夾
from pathlib import Path
from urllib.parse import parse_qs, urlparse
from datetime import datetime
from yt_dlp import YoutubeDL
import re
import os

# --- 全域配置 ---
MODEL_SIZE = "large-v3"   # tiny / base / small / medium / large-v3
LANGUAGE = "zh"           # None = 自動偵測 zh:中文 en:英文 ja:日語
TASK = "transcribe"       # transcribe:直接輸出 / translate:翻譯成英文後輸出
REDOWNLOAD_AUDIO = False   # True: 強制重新下載 mp3

WORKSPACE_DIR = Path.cwd().resolve()
URLS_FILE = WORKSPACE_DIR / "urls.txt"
BASE_OUTPUT_DIR = (WORKSPACE_DIR / "projects").resolve()
SHARED_MODEL_DIR = (WORKSPACE_DIR / "model_cache").resolve()
SHARED_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# 讀取待處理連結
if not URLS_FILE.exists():
    URLS_FILE.write_text("# 貼上網址\n", encoding="utf-8")
    print(f"請在 {URLS_FILE} 加入網址後再執行。")
    VIDEO_URLS = []
else:
    raw_lines = URLS_FILE.read_text(encoding="utf-8").splitlines()
    VIDEO_URLS = [line.strip() for line in raw_lines if line.strip() and not line.startswith("#")]

print(f"待處理影片數量: {len(VIDEO_URLS)}")

def find_ffmpeg_exe() -> Path | None:
    direct = Path.home() / "AppData/Local/Microsoft/WinGet/Links/ffmpeg.exe"
    if direct.exists():
        return direct.resolve()

    package_root = Path.home() / "AppData/Local/Microsoft/WinGet/Packages"
    if package_root.exists():
        candidates = sorted(package_root.glob("*FFmpeg*/*/bin/ffmpeg.exe"))
        if candidates:
            return candidates[-1].resolve()

    return None

FFMPEG_EXE = find_ffmpeg_exe()
if not FFMPEG_EXE:
    raise FileNotFoundError("找不到 ffmpeg.exe，請確認已安裝 Gyan.FFmpeg")

def extract_video_id(url: str) -> str:
    parsed = urlparse(url)
    if "youtube.com" in parsed.netloc:
        return parse_qs(parsed.query).get("v", ["video"])[0]
    if "youtu.be" in parsed.netloc:
        return parsed.path.strip("/") or "video"
    return re.sub(r"[^a-zA-Z0-9_-]", "_", parsed.path.strip("/")) or "video"

def sanitize_windows_name(name: str, max_len: int = 60) -> str:
    cleaned = re.sub(r"[<>:\"/\\|?*]", "_", name).strip().rstrip(".")
    cleaned = re.sub(r"\s+", " ", cleaned)
    return (cleaned[:max_len].strip() or "untitled")

def fetch_video_title(url: str) -> str:
    ydl_opts = {"quiet": True, "no_warnings": True, "skip_download": True}
    try:
        with YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=False)
            return (info or {}).get("title") or "untitled"
    except Exception as e:
        print(f"⚠️ 無法取得影片標題，將使用 untitled: {e}")
        return "untitled"


In [ ]:
# 批次處理邏輯
import subprocess
import sys
import os
from tqdm.auto import tqdm
from faster_whisper import WhisperModel
from huggingface_hub import snapshot_download

# --- 準備模型 ---
def resolve_repo_id(model_size: str) -> str:
    if "/" in model_size: return model_size
    return f"Systran/faster-whisper-{model_size}"

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
repo_id = resolve_repo_id(MODEL_SIZE)
local_model_dir = (SHARED_MODEL_DIR / repo_id.replace("/", "__")).resolve()
local_model_dir.mkdir(parents=True, exist_ok=True)

print("正在檢查/下載模型...")
model_path = snapshot_download(repo_id=repo_id, local_dir=str(local_model_dir))
model = WhisperModel(model_path, compute_type="float16")

# --- 開始循環處理 ---
for idx, VIDEO_URL in enumerate(VIDEO_URLS, 1):
    print(f"\n--- [{idx}/{len(VIDEO_URLS)}] 處理中: {VIDEO_URL} ---")
    
    video_title = fetch_video_title(VIDEO_URL)
    safe_title = sanitize_windows_name(video_title)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    PROJECT_DIR = (BASE_OUTPUT_DIR / f"{timestamp}_{safe_title}").resolve()
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    
    AUDIO_PATH = PROJECT_DIR / "input.mp3"
    OUTPUT_TXT_PATH = PROJECT_DIR / "output.txt"
    OUTPUT_SRT_PATH = PROJECT_DIR / "output.srt"
    
    (PROJECT_DIR / "source.url").write_text(f"[InternetShortcut]\nURL={VIDEO_URL}\n", encoding="utf-8")

    # 1. 下載音訊
    if AUDIO_PATH.exists() and not REDOWNLOAD_AUDIO:
        print(f"ℹ️ 已有既存音訊，略過下載")
    else:
        cmd = [sys.executable, "-m", "yt_dlp", "--no-update", "--ffmpeg-location", str(FFMPEG_EXE),
               "-x", "--audio-format", "mp3", "-o", str(AUDIO_PATH), VIDEO_URL]
        subprocess.run(cmd, capture_output=True, text=True)

    if not AUDIO_PATH.exists():
        print(f"❌ 無法取得音訊，跳過此影片")
        continue

    # 2. 轉錄
    segments_iter, _ = model.transcribe(str(AUDIO_PATH), beam_size=5, vad_filter=True, language=LANGUAGE, task=TASK)
    
    segments = []
    for seg in tqdm(segments_iter, desc="轉錄中", leave=False):
        segments.append(seg)

    # 3. 儲存結果
    with open(OUTPUT_TXT_PATH, "w", encoding="utf-8") as f:
        for seg in segments: f.write(seg.text.strip() + "\n")
    
    def format_time(t):
        h, m, s = int(t//3600), int((t%3600)//60), int(t%60)
        ms = int((t-int(t))*1000)
        return f"{h:02}:{m:02}:{s:02},{ms:03}"

    with open(OUTPUT_SRT_PATH, "w", encoding="utf-8") as f:
        for i, seg in enumerate(segments, 1):
            f.write(f"{i}\n{format_time(seg.start)} --> {format_time(seg.end)}\n{seg.text.strip()}\n\n")

    print(f"✅ 完成: {safe_title}")

print("\n✨ 所有任務處理完畢！")
